In [64]:
import os
os.environ["OPENAI_API_KEY"] = "Api-key"

In [2]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.8 MB/s eta 0:00:00


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
# tool create

@tool
def multiply(a: int, b:int) -> int:
    """Given 2 numbers a and b this tool return their product"""
    return a*b

In [5]:
print(multiply.invoke({"a": 4, "b": 7}))

28


In [6]:
multiply.name

'multiply'

In [7]:
multiply.description

'Given 2 numbers a and b this tool return their product'

In [8]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [9]:
# tool binding
llm = ChatOpenAI(model="gpt-5-nano-2025-08-07")

In [10]:
llm_with_tools = llm.bind_tools([multiply])

In [11]:
llm_with_tools.invoke("Hi, how are you")

AIMessage(content='Hi! I’m here and ready to help. How can I assist you today? If you have a task in mind, feel free to share the details.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 233, 'prompt_tokens': 139, 'total_tokens': 372, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DlBymd7clWObBVDzhhtN87mM0rQRS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e789a-3c69-7070-8e9e-096d861cfa61-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 139, 'output_tokens': 233, 'total_tokens': 372, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 192}})

In [12]:
query = HumanMessage('can you multiply 3 with 10')

In [13]:
messages = [query]

In [14]:
messages

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={})]

In [15]:
result = llm_with_tools.invoke(messages)

In [16]:
messages.append(result)

In [17]:
messages

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 282, 'prompt_tokens': 142, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DlBypT9KuiUZuQTgb7QebueEaJ4E5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e789a-4dff-71c0-b236-9f47e557fcf8-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_1gCqJrzTzlJ6r9EZjeZiLi9i', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 142, 'output_tokens': 282, 'total_tokens': 424, 'input_token_details': {'audio': 

In [18]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'call_1gCqJrzTzlJ6r9EZjeZiLi9i',
 'type': 'tool_call'}

In [19]:
result.tool_calls[0]['args']

{'a': 3, 'b': 10}

In [20]:
multiply.invoke(result.tool_calls[0]['args'])

30

In [21]:
# for tool message
multiply.invoke(result.tool_calls[0])

ToolMessage(content='30', name='multiply', tool_call_id='call_1gCqJrzTzlJ6r9EZjeZiLi9i')

In [22]:
tool_result = multiply.invoke(result.tool_calls[0])
messages.append(tool_result)

In [23]:
messages

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 282, 'prompt_tokens': 142, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DlBypT9KuiUZuQTgb7QebueEaJ4E5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e789a-4dff-71c0-b236-9f47e557fcf8-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_1gCqJrzTzlJ6r9EZjeZiLi9i', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 142, 'output_tokens': 282, 'total_tokens': 424, 'input_token_details': {'audio': 

In [24]:
llm_with_tools.invoke(messages).content

'3 multiplied by 10 equals 30.'

# **Currency conversion - example project**

In [48]:
# tool create

from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """

    url = f"https://v6.exchangerate-api.com/v6/5f1624922351ac0d3a1b4534/pair/{base_currency}/{target_currency}"

    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_currency_value * conversion_rate

In [49]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1780099202,
 'time_last_update_utc': 'Sat, 30 May 2026 00:00:02 +0000',
 'time_next_update_unix': 1780185602,
 'time_next_update_utc': 'Sun, 31 May 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.1992}

In [50]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [51]:
# tool binding 
llm = ChatOpenAI()

In [52]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [53]:
llm_with_tools

_ChatModelBinding(bound=ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7cfcfe77f350>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7cfe1b6d0380>, root_client=<openai.OpenAI object at 0x7cfcfe77f230>, root_async_client=<openai.AsyncOpenAI object at 0x7cfcfe77d430>, model_kwargs={}, openai_api_key=SecretStr('*****

In [54]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [55]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [56]:
ai_message = llm_with_tools.invoke(messages)

In [57]:
ai_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 123, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DlCJt4uM6ZabZoVqZYNNuIOZeNLsx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e78ae-3d2b-7533-94c2-405762cba86d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_iRFlbnD7uZr6js4o0CO1wJqN', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'call_FiNawZWNko3nzI1NTrnEAlTf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 123, 'output_tokens':

In [58]:
messages.append(ai_message)

In [59]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_iRFlbnD7uZr6js4o0CO1wJqN',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_FiNawZWNko3nzI1NTrnEAlTf',
  'type': 'tool_call'}]

In [61]:
import json
for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)

        # fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']

        #append this tool message to messages list
        messages.append(tool_message1)

    #execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name'] == 'convert':

        # fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [62]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 123, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DlCJt4uM6ZabZoVqZYNNuIOZeNLsx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e78ae-3d2b-7533-94c2-405762cba86d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_iRFlbnD7uZr6js4o0CO1wJqN', 'type': 'tool_call'}, {'name': 'convert', 'args

In [63]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 95.1992. Therefore, 10 USD is equal to 951.99 INR.'